In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler,StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split,KFold,GridSearchCV
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_selector,make_column_transformer

In [11]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878) 
  
# data (as pandas dataframes) 
X = cirrhosis_patient_survival_prediction.data.features 

y = cirrhosis_patient_survival_prediction.data.targets 

In [3]:
X=X.replace('NaNN',np.nan)
X.Cholesterol=pd.to_numeric(X.Cholesterol,errors='coerce')
X.Copper=pd.to_numeric(X.Copper,errors='coerce')
X.Tryglicerides=pd.to_numeric(X.Tryglicerides,errors='coerce')
X.Platelets=pd.to_numeric(X.Platelets,errors='coerce')

In [4]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Drug           312 non-null    str    
 1   Age            418 non-null    int64  
 2   Sex            418 non-null    str    
 3   Ascites        312 non-null    str    
 4   Hepatomegaly   312 non-null    str    
 5   Spiders        312 non-null    str    
 6   Edema          418 non-null    str    
 7   Bilirubin      418 non-null    float64
 8   Cholesterol    284 non-null    float64
 9   Albumin        418 non-null    float64
 10  Copper         310 non-null    float64
 11  Alk_Phos       312 non-null    float64
 12  SGOT           312 non-null    float64
 13  Tryglicerides  282 non-null    float64
 14  Platelets      407 non-null    float64
 15  Prothrombin    416 non-null    float64
 16  Stage          412 non-null    float64
dtypes: float64(10), int64(1), str(6)
memory usage: 60.9 KB


In [5]:
X.Drug.value_counts()

Drug
D-penicillamine    158
Placebo            154
Name: count, dtype: int64

In [6]:
imp_cat=SimpleImputer(strategy='most_frequent')
imp_num=SimpleImputer(strategy='mean')
ct_imp=make_column_transformer((imp_cat,make_column_selector(dtype_include=object)),(imp_num, make_column_selector(dtype_exclude= object)),
                                verbose_feature_names_out = False).set_output(transform='pandas')

ohe=OneHotEncoder(drop='first',sparse_output=False).set_output(transform='pandas')

ct_ohe=make_column_transformer((ohe,make_column_selector(dtype_include=object)),
                               ('passthrough',make_column_selector(dtype_exclude=object)),verbose_feature_names_out=False)
ss=StandardScaler()

mm=MinMaxScaler()

kf=KFold(shuffle=True, random_state=26)

knn=KNeighborsClassifier()

pipe=Pipeline([('IMP',ct_imp),('OHE',ct_ohe),('SC',ss),('KNN',knn)])

In [7]:
pipe.get_params()

{'memory': None,
 'steps': [('IMP',
   ColumnTransformer(transformers=[('simpleimputer-1',
                                    SimpleImputer(strategy='most_frequent'),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000001CFF69EBE00>),
                                   ('simpleimputer-2', SimpleImputer(),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000001CFF6927250>)],
                     verbose_feature_names_out=False)),
  ('OHE',
   ColumnTransformer(transformers=[('onehotencoder',
                                    OneHotEncoder(drop='first',
                                                  sparse_output=False),
                                    <sklearn.compose._column_transformer.make_column_selector object at 0x000001CFF6A31D10>),
                                   ('passthrough', 'passthrough',
                                    <sklearn.compose._column

In [10]:
params={
    'IMP__simpleimputer-1__strategy':['constant','most_frequent'],
    'IMP__simpleimputer-2__strategy':[ 'mean','median'],
    'SC':[ss,mm,None],
    'KNN__n_neighbors':np.arange(1,15)
}
gcv=GridSearchCV(pipe,param_grid=params,scoring='neg_log_loss',cv=kf,n_jobs=-1)
gcv.fit(X,y)
print('Best params:', gcv.best_params_)
print('Best score:', gcv.best_score_)


Best params: {'IMP__simpleimputer-1__strategy': 'constant', 'IMP__simpleimputer-2__strategy': 'median', 'KNN__n_neighbors': np.int64(14), 'SC': None}
Best score: -1.3151709681831885


C:\Users\PGCP-AI.STUDENTSDC\AppData\Local\anaconda3\Lib\site-packages\sklearn\neighbors\_classification.py:243: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
